# Species Classifier — CNN Training Notebook

**Architecture:** 4× Conv2D -> BatchNorm -> MaxPool -> Flatten -> Dense(256) -> Dropout(0.5) -> Softmax(12)  
**Tracking:** MLflow - TensorBoard  
**Model registry:** MLflow only — no local export

## 0. Setup & Configuration

In [ ]:
import os
import json
import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for saving plots
import matplotlib.pyplot as plt

# -- Paths -------------------------------------------------------------
# BASE_DIR is always fastapi/ — the folder this notebook lives in.
import pathlib as _pl

_here = _pl.Path(os.path.abspath('')).resolve()

_root = None
for _candidate in [_here] + list(_here.parents)[:3]:
    if (_candidate / 'data').is_dir():
        _root = _candidate
        break

if _root is None:
    _root = _here

BASE_DIR = str(_root)

assert os.path.isdir(os.path.join(BASE_DIR, 'data')), (
    f"BASE_DIR={BASE_DIR!r} — data/ not found.\n"
    f"Manually set BASE_DIR at the top of this cell to your fastapi/ path."
)

DATA_DIR     = os.path.join(BASE_DIR, "data")
LOG_DIR_BASE = os.path.join(BASE_DIR, "logs", "fit")

# Input data
IMAGES_PATH = os.path.join(DATA_DIR, "images_plant.npy")
LABELS_PATH = os.path.join(DATA_DIR, "Labels_plant.csv")

# Per-version artefact paths — version set after MLflow registry check in Cell 13
# Placeholder paths updated once next_version is known
PRED_PROBS_PATH = None
Y_TEST_PATH     = None

os.makedirs(DATA_DIR,     exist_ok=True)
os.makedirs(LOG_DIR_BASE, exist_ok=True)

print(f"BASE_DIR : {BASE_DIR}")
print(f"DATA_DIR : {DATA_DIR}")

## 1. Load Data

In [ ]:
df     = np.load(IMAGES_PATH)
labels = pd.read_csv(LABELS_PATH)

print(f'Images shape : {df.shape}')
print(f'Labels shape : {labels.shape}')
print(f'Unique labels: {labels.nunique().values[0]}')

## 2. Dataset Preparation

In [ ]:
# import psutil
# ram = psutil.virtual_memory()
# print(f"Available : {ram.available / 1024**3:.1f} GB")
# print(f"Total     : {ram.total / 1024**3:.1f} GB")
# print(f"Used      : {ram.used / 1024**3:.1f} GB")

In [ ]:
from sklearn.preprocessing import LabelEncoder

labels    = labels.squeeze()
le        = LabelEncoder()
y_encoded = le.fit_transform(labels)

print(f'Number of classes: {len(le.classes_)}')
print(f'Classes: {list(le.classes_)}')
print(pd.Series(y_encoded).value_counts())

# -- In-place normalisation — avoids creating a full copy -----------------------
# Skip conversion if already float32
if df.dtype == 'float32':
    X = df / 255.0
else:
    X = df.astype('float32') / 255.0
del df
import gc; gc.collect()

print(f"X shape: {X.shape}  dtype: {X.dtype}  size: {X.nbytes/1024**3:.2f} GB")

## 3. Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

## 4. Model Architecture

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Conv2D(32,  (3,3), activation='relu', input_shape=(128,128,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64,  (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(12, activation='softmax')
], name='species_cnn')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Callbacks (TensorBoard - EarlyStopping - Checkpoint)

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# --- MLflow setup --------------------------------------------------------------
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("species_cnn_classifier")

# --- Auto-increment version from MLflow registry ------------------------------
client = MlflowClient()
try:
    all_versions = client.search_model_versions("name='cnn-species-classifier'")
    next_version = len(all_versions) + 1
except Exception:
    next_version = 1

# -- Per-version artefact directory --------------------------------------------
VERSION_DIR     = os.path.join(DATA_DIR, f"v{next_version}")
PRED_PROBS_PATH = os.path.join(VERSION_DIR, "cnn_pred_probs.npy")
Y_TEST_PATH     = os.path.join(VERSION_DIR, "cnn_y_test.npy")
os.makedirs(VERSION_DIR, exist_ok=True)

# --- Timestamped TensorBoard log directory --------------------------------------
run_ts  = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
log_dir = os.path.join(LOG_DIR_BASE, f'v{next_version}_{run_ts}')
os.makedirs(log_dir, exist_ok=True)

# -- Callbacks -------------------------------------------------------------------
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir, histogram_freq=1, write_graph=True
)

early_stop = tf.keras.callbacks.EarlyStopping(
    patience=5, restore_best_weights=True
)

# Temp checkpoint — deleted after MLflow registration
best_model_path = os.path.join(BASE_DIR, "temp_best_model.keras")
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    best_model_path, monitor='val_loss', save_best_only=True
)

# -- LR Plateau — reduce LR when val_loss stops improving ---------------------------
lr_plateau = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',   # watch validation loss
    factor=0.5,           # multiply LR by 0.5 when triggered
    patience=3,           # wait 3 epochs before reducing
    min_lr=1e-6,          # never go below this LR
    verbose=1             # print message when LR is reduced
)

print(f'MLflow URI         : {mlflow.get_tracking_uri()}')
print(f'Next version       : v{next_version} (auto from MLflow registry)')
print(f'TensorBoard log dir: {log_dir}')
print(f'Temp checkpoint    : {best_model_path}')
print()
print('Launch TensorBoard (run in a terminal from fastapi/ folder):')
print('  .venv\\Scripts\\python.exe -m tensorboard.main --logdir logs/fit')

## 6. Training

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[early_stop, tensorboard_callback, checkpoint, lr_plateau]
)

## 7. Training Curves

In [ ]:
# Loss curve
fig_loss, ax = plt.subplots(figsize=(8, 5))
ax.plot(history.history['loss'],     label='Train Loss')
ax.plot(history.history['val_loss'], label='Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training vs Validation Loss')
ax.legend(); plt.tight_layout(); plt.show()

# Accuracy curve
fig_acc, ax = plt.subplots(figsize=(8, 5))
ax.plot(history.history['accuracy'],     label='Train Accuracy')
ax.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Training vs Validation Accuracy')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Evaluation (Train / Val / Test)

In [ ]:
print('Train Evaluation')
train_loss, train_acc = model.evaluate(X_train, y_train)

print('\nValidation Evaluation')
val_loss, val_acc = model.evaluate(X_val, y_val)

print('\nTest Evaluation')
test_loss, test_acc = model.evaluate(X_test, y_test)

print(f'\nSummary — Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f} | Test acc: {test_acc:.4f}')

## 9. Classification Report & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = model.predict(X_test)
y_pred       = np.argmax(y_pred_probs, axis=1)

# report_dict used by MLflow artefacts block
report_dict = classification_report(
    y_test, y_pred, target_names=le.classes_, output_dict=True
)
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:\n', cm)

## 10. MLflow — Log, Register & Promote

In [ ]:
import mlflow.keras
import tempfile

with mlflow.start_run(run_name=f"cnn-v{next_version}_{run_ts}") as run:

    # -- Parameters -------------------------------------------------------
    mlflow.log_params({
        "model_version":       next_version,
        "architecture":        model.name,
        "framework":           "TensorFlow/Keras",
        "optimizer":           "adam",
        "loss":                "sparse_categorical_crossentropy",
        "epochs_max":          40,
        "epochs_trained":      len(history.history["loss"]),
        "batch_size":          32,
        "early_stop_patience": 5,
        "lr_plateau_factor":   0.5,
        "lr_plateau_patience": 3,
        "lr_plateau_min_lr":   1e-6, 
        "input_shape":         "128x128x3",
        "num_classes":         len(le.classes_),
        "train_size":          X_train.shape[0],
        "val_size":            X_val.shape[0],
        "test_size":           X_test.shape[0],
    })

    # -- Final metrics ------------------------------------------------------
    mlflow.log_metrics({
        "train_accuracy": round(float(train_acc),  4),
        "val_accuracy":   round(float(val_acc),    4),
        "test_accuracy":  round(float(test_acc),   4),
        "train_loss":     round(float(train_loss), 4),
        "val_loss":       round(float(val_loss),   4),
        "test_loss":      round(float(test_loss),  4),
    })

    # --Per-epoch metrics -----------------------------------------------------
    for epoch, (tl, ta, vl, va) in enumerate(zip(
        history.history["loss"],
        history.history["accuracy"],
        history.history["val_loss"],
        history.history["val_accuracy"],
    )):
        mlflow.log_metrics({
            "epoch_train_loss":     tl,
            "epoch_train_accuracy": ta,
            "epoch_val_loss":       vl,
            "epoch_val_accuracy":   va,
        }, step=epoch)

    # -- 1. Training history --------------------------------------------------
    history_serialisable = {k: [float(v) for v in vals]
                            for k, vals in history.history.items()}
    _history_tmp = os.path.join(tempfile.gettempdir(), "cnn_history.json")
    with open(_history_tmp, "w") as f:
        json.dump(history_serialisable, f, indent=2)
    mlflow.log_artifact(_history_tmp, artifact_path="artefacts")
    print("Training history logged to MLflow")

    # -- 2. Model info card -------------------------------------------------
    model_info = {
        "version":        next_version,
        "architecture":   model.name,
        "framework":      "TensorFlow/Keras",
        "trained_on":     str(pd.Timestamp.now().date()),
        "input_shape":    [128, 128, 3],
        "num_classes":    int(len(le.classes_)),
        "classes":        list(le.classes_),
        "epochs_trained": len(history.history["loss"]),
        "mlflow_run_id":  run.info.run_id,
        "mlflow_model":   "cnn-species-classifier",
        "mlflow_stage":   "Production",
        "tensorboard_logdir": log_dir,
        "train_acc":      round(float(train_acc),  4),
        "val_acc":        round(float(val_acc),    4),
        "test_acc":       round(float(test_acc),   4),
        "train_loss":     round(float(train_loss), 4),
        "val_loss":       round(float(val_loss),   4),
        "test_loss":      round(float(test_loss),  4),
        "intended_use":   "Species classification from plant images",
        "input":          "RGB image 128x128",
        "output":         "Species class + confidence score",
        "limitations":    "Trained on limited dataset — may not generalise to unseen species",
    }
    _card_tmp = os.path.join(tempfile.gettempdir(), "cnn_model_info.json")
    with open(_card_tmp, "w") as f:
        json.dump(model_info, f, indent=2)
    mlflow.log_artifact(_card_tmp, artifact_path="model_card")
    print("Model info card logged to MLflow")

    # -- 3. Classification report ------------------------------------------------
    _clf_tmp = os.path.join(tempfile.gettempdir(), "cnn_classification_report.csv")
    pd.DataFrame(report_dict).T.to_csv(_clf_tmp)
    mlflow.log_artifact(_clf_tmp, artifact_path="artefacts")
    print("Classification report logged to MLflow")

    # -- 4. Confusion matrix ------------------------------------------------------
    _cm_tmp = os.path.join(tempfile.gettempdir(), "cnn_confusion_matrix.csv")
    pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_csv(_cm_tmp)
    mlflow.log_artifact(_cm_tmp, artifact_path="artefacts")
    print("Confusion matrix logged to MLflow")

    # -- 5. Predicted probabilities & ground truth ----------------------------------
    # Kept locally so Streamlit can read .npy files directly from mounted volume
    np.save(PRED_PROBS_PATH, y_pred_probs)
    np.save(Y_TEST_PATH,     y_test)
    mlflow.log_artifact(str(PRED_PROBS_PATH), artifact_path="artefacts")
    mlflow.log_artifact(str(Y_TEST_PATH),     artifact_path="artefacts")
    print(f"Pred probs + y_test -> {VERSION_DIR} (local + MLflow)")

    # ------ 6. Label classes ------------------------------------------------------
    _labels_tmp = os.path.join(tempfile.gettempdir(), "cnn_label_classes.json")
    with open(_labels_tmp, "w") as f:
        json.dump(list(le.classes_), f)
    mlflow.log_artifact(_labels_tmp, artifact_path="labels")
    print("Label classes logged to MLflow")

    # -- 7. Training curve plots --------------------------------------------------
    fig_loss.savefig(os.path.join(tempfile.gettempdir(), "cnn_loss_curve.png"))
    fig_acc.savefig(os.path.join(tempfile.gettempdir(),  "cnn_accuracy_curve.png"))
    mlflow.log_artifact(os.path.join(tempfile.gettempdir(), "cnn_loss_curve.png"),     artifact_path="plots")
    mlflow.log_artifact(os.path.join(tempfile.gettempdir(), "cnn_accuracy_curve.png"), artifact_path="plots")
    print("Training curve plots logged to MLflow")

    # -- 8. Register model in MLflow ----------------------------------------------
    mlflow.keras.log_model(
        model,
        name="cnn-model",
        registered_model_name="cnn-species-classifier"
    )
    print("CNN model registered in MLflow")

    # -- 9. Auto-delete temp checkpoint -------------------------------------
    if os.path.exists(best_model_path):
        os.remove(best_model_path)
        print(f"Temp checkpoint deleted: {best_model_path}")


    # -- 10. Verify all artifacts uploaded ------------------------------------
    print("\nVerifying MLflow artifacts...")
    try:
        artifacts = mlflow.artifacts.list_artifacts(run_id=run.info.run_id)
        uploaded  = {a.path for a in artifacts}

        required = [
            "artefacts/cnn_history.json",
            "model_card/cnn_model_info.json",
            "artefacts/cnn_classification_report.csv",
            "artefacts/cnn_confusion_matrix.csv",
            "artefacts/cnn_pred_probs.npy",
            "artefacts/cnn_y_test.npy",
            "labels/cnn_label_classes.json",
        ]

        all_ok = True
        for f in required:
            if f in uploaded:
                print(f"  [OK]      {f}")
            else:
                print(f"  [MISSING] {f} ← not uploaded!")
                all_ok = False

        if all_ok:
            print("\nAll CNN artifacts verified ✅")
        else:
            print("\n[WARN] Some CNN artifacts missing — check logs above")
    except Exception as e:
        print(f"[WARN] Could not verify artifacts: {e}")

    run_id = run.info.run_id

# -- Promote to Production ---------------------------------------------------------
versions = client.get_latest_versions("cnn-species-classifier", stages=["None"])
if versions:
    client.transition_model_version_stage(
        name="cnn-species-classifier",
        version=versions[0].version,
        stage="Production"
    )
    print(f"CNN model v{versions[0].version} promoted to Production")
else:
    print("[WARN] No new versions found to promote — already in Production?")

print()
print("================================================")
print(f"  CNN Training Complete")
print(f"  MLflow model  : models:/cnn-species-classifier/Production")
print(f"  MLflow run    : {run_id}")
print(f"  TensorBoard   : {log_dir}")
print(f"  Test accuracy : {test_acc:.4f}")
print(f"  Version       : v{next_version} (auto from MLflow registry)")
print(f"  View MLflow   : http://localhost:5000")
print("================================================")

## 11. Quick Prediction Test

In [ ]:
sample = X_test[11:12]
probs  = model.predict(sample)[0]
for name, p in sorted(zip(le.classes_, probs), key=lambda x: -x[1]):
    print(f"{name:30s}  {p:.4f}  {'█' * int(p * 40)}")
print(f"\nTrue label: {le.classes_[y_test[11]]}")

## 12. Optional: 5-Fold Cross-Validation
Set `RUN_CV = True` to run stratified k-fold CV on the full dataset.  
**Takes ~2–3× the normal training time** — run once for evaluation, not every run.

In [ ]:
RUN_CV   = False
CV_FOLDS = 5

if RUN_CV:
    import time
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import f1_score

    print(f'Running {CV_FOLDS}-fold stratified cross-validation...')
    X_cv = np.concatenate([X_train, X_val, X_test], axis=0)
    y_cv = np.concatenate([y_train, y_val,  y_test], axis=0)

    skf        = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    cv_results = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cv, y_cv), 1):
        t0 = time.time()
        print(f'Fold {fold}/{CV_FOLDS} — training...')

        X_f_train, X_f_val = X_cv[train_idx], X_cv[val_idx]
        y_f_train, y_f_val = y_cv[train_idx], y_cv[val_idx]

        fold_model = keras.Sequential([
            layers.Conv2D(32,  (3,3), activation='relu', input_shape=(128,128,3)),
            layers.BatchNormalization(), layers.MaxPooling2D(),
            layers.Conv2D(64,  (3,3), activation='relu'),
            layers.BatchNormalization(), layers.MaxPooling2D(),
            layers.Conv2D(128, (3,3), activation='relu'),
            layers.BatchNormalization(), layers.MaxPooling2D(),
            layers.Conv2D(256, (3,3), activation='relu'),
            layers.MaxPooling2D(),
            layers.Flatten(),
            layers.Dense(256, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(len(le.classes_), activation='softmax'),
        ], name=f'cv_fold_{fold}')
        fold_model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        fold_history = fold_model.fit(
            X_f_train, y_f_train,
            validation_data=(X_f_val, y_f_val),
            epochs=40, batch_size=32,
            callbacks=[keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=5, restore_best_weights=True
            )],
            verbose=0,
        )

        fold_loss, fold_acc = fold_model.evaluate(X_f_val, y_f_val, verbose=0)
        y_f_pred = np.argmax(fold_model.predict(X_f_val, verbose=0), axis=1)
        fold_f1  = f1_score(y_f_val, y_f_pred, average='weighted')
        epochs_ran = len(fold_history.history['loss'])

        cv_results.append({
            'fold': fold, 'accuracy': round(float(fold_acc), 4),
            'f1':   round(float(fold_f1), 4), 'loss': round(float(fold_loss), 4),
            'epochs': epochs_ran,
        })
        print(f'  Fold {fold}: acc={fold_acc:.4f}  f1={fold_f1:.4f}  '
              f'loss={fold_loss:.4f}  epochs={epochs_ran}  ({(time.time()-t0)/60:.1f} min)')

    accs = [r['accuracy'] for r in cv_results]
    f1s  = [r['f1']       for r in cv_results]

    cv_summary = {
        'n_folds': CV_FOLDS,
        'mean_accuracy': round(float(np.mean(accs)), 4),
        'std_accuracy':  round(float(np.std(accs)),  4),
        'mean_f1':       round(float(np.mean(f1s)),  4),
        'std_f1':        round(float(np.std(f1s)),   4),
        'folds':         cv_results,
    }

    # Save locally for Streamlit + log to MLflow
    cv_path = os.path.join(VERSION_DIR, 'cnn_cv_results.json')
    with open(cv_path, 'w') as f:
        json.dump(cv_summary, f, indent=2)

    # Log to MLflow if a run is active
    try:
        mlflow.log_artifact(cv_path, artifact_path="artefacts")
        print("CV results logged to MLflow")
    except Exception:
        print(f"CV results saved locally: {cv_path}")

    print()
    print(f'{CV_FOLDS}-Fold CV Summary')
    print(f'  Mean Accuracy : {np.mean(accs):.4f} ± {np.std(accs):.4f}')
    print(f'  Mean F1       : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
    for r in cv_results:
        print(f'Fold {r["fold"]}: acc={r["accuracy"]:.4f}  f1={r["f1"]:.4f}  epochs={r["epochs"]}')

else:
    print('CV skipped (RUN_CV=False). Set RUN_CV=True to run cross-validation.')
    cv_summary = None

## 13. TensorBoard

TensorBoard runs as a Docker container and reads logs from `fastapi/logs/fit/`.  
Open: **http://localhost:6006**

- **Scalars tab** → loss & accuracy curves per epoch
- **Graphs tab**  → full CNN architecture graph
- **Histograms tab** → weight distributions per layer

To run locally outside Docker:
```
cd fastapi
.venv\Scripts\python.exe -m tensorboard.main --logdir logs/fit
```